# Image Corruption Evaluation

This notebook evaluates the robustness of the trained baseline CNN under image corruptions. The clean CIFAR-10 test performance is first reproduced as a reference, after which controlled corruptions are applied to the same test images.

Robustness will be evaluated by comparing performance under corrupted inputs with the clean baseline performance.

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader

In [2]:
# CIFAR-10 normalization statistics
mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2435, 0.2616]

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Load Baseline Model

The baseline CNN architecture and the checkpoint selected using the lowest validation loss are restored for robustness evaluation.

In [4]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [6]:
checkpoint_path = Path("../models/baseline_cnn_best.pth")

model = BaselineCNN().to(device)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

## Clean Test Baseline

Before applying image corruptions, the clean test performance is reproduced to verify that the saved checkpoint and evaluation pipeline are working correctly.

In [8]:
clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

clean_test_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=clean_transform
)

clean_test_loader = DataLoader(
    clean_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

In [10]:
criterion = nn.CrossEntropyLoss()

def evaluate_model(model, data_loader, criterion, device):
    running_loss = 0.0
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    average_loss = running_loss / total
    accuracy = correct / total

    return average_loss, accuracy


In [12]:
clean_loss, clean_accuracy = evaluate_model(
    model,
    clean_test_loader,
    criterion,
    device
)

print(f"Clean Test Loss: {clean_loss:.4f}")
print(f"Clean Test Accuracy: {clean_accuracy:.4f}")

Clean Test Loss: 0.8143
Clean Test Accuracy: 0.7152
